## **Import Library**

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ====== PENGATURAN FILE ======
INPUT_FILE  = Path("../data/processed/data_labeled_ags_sep_okt.xlsx")
OUTPUT_DIR  = Path("../data/processed")
OUTPUT_FILE = OUTPUT_DIR / "data_prepared.xlsx"
# =============================

# Baca dari Excel (.xlsx): satu sel = satu menfess, jadi koma atau enter
# di dalam teks tidak akan memecah baris seperti pada CSV.
# Perlu: pip install openpyxl
data = pd.read_excel(
    INPUT_FILE,
    sheet_name=0,              # sheet pertama
    engine="openpyxl",
    dtype={"Text": "string"},  # pastikan teks tidak diubah jadi tipe lain
)

data.columns = data.columns.str.strip()

print("Ukuran data:", data.shape)
display(data.head())
display(data.dtypes)

Ukuran data: (5446, 7)


,Message_ID,Sender_ID,Date_WIB,Text,Is_relevant,topic_category,sentiment_label
0,9530,-1001647181983,2026-08-31 23:37:04,Info kost kost an cowo under 750 yg jarak nya ...,No,NaN,NaN
1,9529,-1001647181983,2026-08-31 22:57:19,Sender punya bisnis yang Alhamdulillah lolos p...,Yes,Organisasi & Pengembangan Mahasiswa,Netral
2,9528,-1001647181983,2026-08-31 22:54:49,"Ara CE maba, sumpah arek e lucu maneh, gak coc...",No,NaN,NaN
3,9527,-1001647181983,2026-08-31 22:53:57,Ada yang jual sepatu futsal gak?,No,NaN,NaN
4,9526,-1001647181983,2026-08-31 22:01:30,"infoin dong oprec DPM kapan ya, sistem pemilih...",Yes,Organisasi & Pengembangan Mahasiswa,Netral


Message_ID                  int64
Sender_ID                   int64
Date_WIB           datetime64[ns]
Text               string[python]
Is_relevant                object
topic_category             object
sentiment_label            object
dtype: object

## **Periksa Missing Value dan Duplikat**

In [2]:
print("Missing value:")
display(data.isna().sum())

print("Baris duplikat identik:", data.duplicated().sum())

print("Teks kosong:")
print(data["Text"].isna().sum())

print("Label relevansi:")
display(data["Is_relevant"].value_counts(dropna=False))

print("Kategori topik:")
display(data["topic_category"].value_counts(dropna=False))

print("Sentimen:")
display(data["sentiment_label"].value_counts(dropna=False))

Missing value:


Message_ID            0
Sender_ID             0
Date_WIB              0
Text                  0
Is_relevant           0
topic_category     3641
sentiment_label    3643
dtype: int64

Baris duplikat identik: 0
Teks kosong:
0
Label relevansi:


Is_relevant
No     3642
Yes    1804
Name: count, dtype: int64

Kategori topik:


topic_category
NaN                                      3641
Layanan & Fasilitas kampus                619
Akademik & Perkuliahan                    316
Layanan & Fasilitas Kampus                281
Organisasi & Pengembangan Mahasiswa       215
Kegiatan & Acara Kampus                   212
Kehidupan dan Kesejahteraan mahasiswa     162
Name: count, dtype: int64

Sentimen:


sentiment_label
NaN        3643
Netral      961
Negatif     765
Positif      77
Name: count, dtype: int64

## **Duplikat berdasarkan teks dan tanggal**

In [3]:
data["Date_WIB"] = pd.to_datetime(
    data["Date_WIB"],
    errors="coerce"
)

data["Text"] = data["Text"].astype("string")

print(
    "Duplikat berdasarkan tanggal dan teks:",
    data.duplicated(
        subset=["Date_WIB", "Text"],
        keep=False
    ).sum()
)

print(
    "Duplikat berdasarkan teks saja:",
    data.duplicated(
        subset=["Text"],
        keep=False
    ).sum()
)

duplicates = data[
    data.duplicated(
        subset=["Date_WIB", "Text"],
        keep=False
    )
].sort_values(["Date_WIB", "Text"])

display(duplicates.head(20))

Duplikat berdasarkan tanggal dan teks: 0
Duplikat berdasarkan teks saja: 14


,Message_ID,Sender_ID,Date_WIB,Text,Is_relevant,topic_category,sentiment_label


## **Menyamakan format label**

In [4]:
for col in [
    "Is_relevant",
    "topic_category",
    "sentiment_label"
]:
    data[col] = data[col].astype("string").str.strip()

data["Is_relevant"] = data["Is_relevant"].str.title()

data["sentiment_label"] = (
    data["sentiment_label"]
    .str.lower()
    .replace({
        "positif": "Positif",
        "positive": "Positif",
        "negatif": "Negatif",
        "negative": "Negatif",
        "netral": "Netral",
        "neutral": "Netral"
    })
)

# Samakan penulisan kategori topik (mis. "Layanan & Fasilitas kampus"
# dan "Layanan & Fasilitas Kampus" sebelumnya terhitung dua kategori berbeda)
TOPIK_KANONIK = {
    "layanan & fasilitas kampus":            "Layanan & Fasilitas Kampus",
    "akademik & perkuliahan":                "Akademik & Perkuliahan",
    "organisasi & pengembangan mahasiswa":   "Organisasi & Pengembangan Mahasiswa",
    "kegiatan & acara kampus":               "Kegiatan & Acara Kampus",
    "kehidupan dan kesejahteraan mahasiswa": "Kehidupan dan Kesejahteraan Mahasiswa",
}
kunci_topik = data["topic_category"].str.lower().str.replace(r"\s+", " ", regex=True)
data["topic_category"] = kunci_topik.map(TOPIK_KANONIK).fillna(data["topic_category"])

# peringatan kalau ada kategori topik di luar daftar di atas
tak_dikenal = set(data["topic_category"].dropna()) - set(TOPIK_KANONIK.values())
if tak_dikenal:
    print("PERHATIAN, kategori topik tidak dikenal:", tak_dikenal)

display(data["Is_relevant"].value_counts(dropna=False))
display(data["sentiment_label"].value_counts(dropna=False))
display(data["topic_category"].value_counts(dropna=False))

Is_relevant
No     3642
Yes    1804
Name: count, dtype: Int64

sentiment_label
<NA>       3643
Netral      961
Negatif     765
Positif      77
Name: count, dtype: Int64

topic_category
<NA>                                     3641
Layanan & Fasilitas Kampus                900
Akademik & Perkuliahan                    316
Organisasi & Pengembangan Mahasiswa       215
Kegiatan & Acara Kampus                   212
Kehidupan dan Kesejahteraan Mahasiswa     162
Name: count, dtype: int64

## **Cek konsistensi label**
Baris yang labelnya saling bertentangan (mis. `Is_relevant = Yes` tapi sentimen kosong). Perlu dicek manual, jadi tidak diubah otomatis.

In [5]:
yes_tanpa_label = data[(data["Is_relevant"] == "Yes") &
                       (data["topic_category"].isna() | data["sentiment_label"].isna())]
no_punya_label  = data[(data["Is_relevant"] == "No") &
                       (data["topic_category"].notna() | data["sentiment_label"].notna())]

print("Relevan (Yes) tapi topik/sentimen kosong:", len(yes_tanpa_label))
display(yes_tanpa_label[["Message_ID", "Is_relevant", "topic_category", "sentiment_label", "Text"]])

print("Tidak relevan (No) tapi punya topik/sentimen:", len(no_punya_label))
display(no_punya_label[["Message_ID", "Is_relevant", "topic_category", "sentiment_label", "Text"]])

Relevan (Yes) tapi topik/sentimen kosong: 1


,Message_ID,Is_relevant,topic_category,sentiment_label,Text
324,9202,Yes,Kehidupan dan Kesejahteraan Mahasiswa,<NA>,"Rek, gimana pendapat kalian ke dosen-dosen yan..."


Tidak relevan (No) tapi punya topik/sentimen: 1


,Message_ID,Is_relevant,topic_category,sentiment_label,Text
1428,8075,No,Kehidupan dan Kesejahteraan Mahasiswa,<NA>,Udah 2026 masa ngga tau trend sih? udah g jama...


## **Kolom bulan (1 = Januari ... 12 = Desember)**

In [6]:
# Pemetaan nomor bulan -> nama bulan (1 = Januari ... 12 = Desember)
BULAN_MAP = {
    1: "Januari",   2: "Februari",  3: "Maret",     4: "April",
    5: "Mei",       6: "Juni",      7: "Juli",      8: "Agustus",
    9: "September", 10: "Oktober",  11: "November", 12: "Desember",
}

data["Date_WIB"] = pd.to_datetime(data["Date_WIB"], errors="coerce")
print("Tanggal tidak valid (NaT):", data["Date_WIB"].isna().sum())

data["month_num"] = data["Date_WIB"].dt.month
data["month"] = data["month_num"].map(BULAN_MAP)

# urutkan bulan secara kronologis, hanya bulan yang benar-benar ada di data
urutan_bulan = [nama for no, nama in BULAN_MAP.items() if no in set(data["month_num"].dropna())]
data["month"] = pd.Categorical(data["month"], categories=urutan_bulan, ordered=True)

rekap_bulan = (
    data.groupby(["month_num", "month"], observed=True)
        .size()
        .rename("jumlah")
        .reset_index()
)
display(rekap_bulan)

Tanggal tidak valid (NaT): 0


,month_num,month,jumlah
0,8,Agustus,1461
1,9,September,2078
2,10,Oktober,1907


In [7]:
print("Jumlah data:", len(data))
print("\nLabel relevansi:")
print(data["Is_relevant"].value_counts(dropna=False))

print("\nLabel sentimen:")
print(data["sentiment_label"].value_counts(dropna=False))

print("\nKategori topik:")
print(data["topic_category"].value_counts(dropna=False))

print("\nPeriode:")
print(data["month"].value_counts(dropna=False, sort=False))

Jumlah data: 5446

Label relevansi:
Is_relevant
No     3642
Yes    1804
Name: count, dtype: Int64

Label sentimen:
sentiment_label
<NA>       3643
Netral      961
Negatif     765
Positif      77
Name: count, dtype: Int64

Kategori topik:
topic_category
<NA>                                     3641
Layanan & Fasilitas Kampus                900
Akademik & Perkuliahan                    316
Organisasi & Pengembangan Mahasiswa       215
Kegiatan & Acara Kampus                   212
Kehidupan dan Kesejahteraan Mahasiswa     162
Name: count, dtype: int64

Periode:
month
Agustus      1461
September    2078
Oktober      1907
Name: count, dtype: int64


## **Simpan hasil (xlsx)**

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Excel menolak karakter kontrol tak terlihat (mis. \x0b). Buang dari kolom Text
# supaya penyimpanan tidak error. Isi teks selain itu tidak diubah.
KARAKTER_ILEGAL = r"[\x00-\x08\x0b\x0c\x0e-\x1f]"
data_out = data.copy()
data_out["Text"] = data_out["Text"].str.replace(KARAKTER_ILEGAL, "", regex=True)
data_out["month"] = data_out["month"].astype("string")

data_out.to_excel(OUTPUT_FILE, index=False, sheet_name="data", engine="openpyxl")

# verifikasi: baca ulang, jumlah baris harus sama
cek = pd.read_excel(OUTPUT_FILE, engine="openpyxl")
print("Tersimpan:", OUTPUT_FILE)
print("Baris asli:", len(data_out), "| baris terbaca ulang:", len(cek))

Tersimpan: ..\data\processed\data_prepared.xlsx
Baris asli: 5446 | baris terbaca ulang: 5446
